# Data Cleaning and Integration

This notebook prepares the selected dissertation datasets for integration into a single analysis-ready dataset.

The datasets used are:
- NESO Historic Demand Data 2023
- Carbon Intensity API data 2023
- ONS System Average Price of Gas data 2023

The aim of this notebook is to clean the datasets, align their time frequency, handle missing data, create useful modelling features, and save a final integrated dataset for exploratory analysis and initial modelling.

## 1. Import Libraries and Set Folder Paths

This section imports the Python libraries used in the notebook and defines the main project folder paths for the raw data, processed data, output tables and output figures.

In [53]:
# Import the libraries needed for data cleaning and integration

import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

In [54]:
# Set the main project folder paths

project_folder = Path.cwd().parent

raw_data_folder = project_folder / "Data" / "raw"
processed_data_folder = project_folder / "Data" / "processed"
outputs_folder = project_folder / "Outputs"
tables_folder = outputs_folder / "tables"
figures_folder = outputs_folder / "figures"

print("Project folder:")
print(project_folder)

print("\nRaw data folder:")
print(raw_data_folder)

print("\nProcessed data folder:")
print(processed_data_folder)

Project folder:
/Users/tosinjimoh/Downloads/Msc/Lecture Notes/2026-MSc-Dissertation-Project

Raw data folder:
/Users/tosinjimoh/Downloads/Msc/Lecture Notes/2026-MSc-Dissertation-Project/Data/raw

Processed data folder:
/Users/tosinjimoh/Downloads/Msc/Lecture Notes/2026-MSc-Dissertation-Project/Data/processed


## 2. Load the Processed Datasets

This section loads the cleaned datasets created during the data source review stage. These datasets will be checked before they are aligned and integrated into a single daily dataset.

In [55]:
# Load the processed datasets for integration

neso_daily_file = processed_data_folder / "neso_demand_wind_solar_2023_daily.csv"
carbon_clean_file = processed_data_folder / "carbon_intensity_2023_clean.csv"
gas_clean_file = processed_data_folder / "ons_sap_gas_2023_clean.csv"

neso_daily = pd.read_csv(neso_daily_file)
carbon_clean = pd.read_csv(carbon_clean_file)
gas_clean = pd.read_csv(gas_clean_file)

print("NESO daily data:")
print(neso_daily.shape)

print("\nCarbon intensity data:")
print(carbon_clean.shape)

print("\nONS gas price data:")
print(gas_clean.shape)

NESO daily data:
(365, 4)

Carbon intensity data:
(17429, 5)

ONS gas price data:
(351, 3)


## 3. Check Date Formats and Data Types

This section checks the column names and data types of the loaded datasets. The date and datetime columns are then converted into proper pandas datetime formats so that the datasets can be aligned and merged correctly.

In [56]:
# Check column names and data types for the loaded datasets

print("NESO daily columns and data types:")
print(neso_daily.dtypes)

print("\nCarbon intensity columns and data types:")
print(carbon_clean.dtypes)

print("\nONS gas price columns and data types:")
print(gas_clean.dtypes)

NESO daily columns and data types:
date                 object
national_demand     float64
wind_generation     float64
solar_generation    float64
dtype: object

Carbon intensity columns and data types:
from                   object
to                     object
forecast_intensity      int64
actual_intensity      float64
intensity_index        object
dtype: object

ONS gas price columns and data types:
date                                   object
sap_actual_p_per_kwh                  float64
sap_7day_rolling_average_p_per_kwh    float64
dtype: object


In [57]:
# Convert date and datetime columns into proper datetime format

neso_daily["date"] = pd.to_datetime(neso_daily["date"])

carbon_clean["from"] = pd.to_datetime(carbon_clean["from"])
carbon_clean["to"] = pd.to_datetime(carbon_clean["to"])

gas_clean["date"] = pd.to_datetime(gas_clean["date"])

print("NESO daily date type:")
print(neso_daily["date"].dtype)

print("\nCarbon intensity from/to date types:")
print(carbon_clean[["from", "to"]].dtypes)

print("\nONS gas price date type:")
print(gas_clean["date"].dtype)

NESO daily date type:
datetime64[ns]

Carbon intensity from/to date types:
from    datetime64[ns, UTC]
to      datetime64[ns, UTC]
dtype: object

ONS gas price date type:
datetime64[ns]


## 4. Prepare Carbon Intensity Data for Daily Integration

This section prepares the Carbon Intensity API dataset for integration with the daily NESO and ONS gas datasets. The Carbon Intensity API data is originally half-hourly, so it is converted to daily average forecast and actual carbon intensity values. Date coverage and missing values are then checked before merging.

In [58]:
# Create a daily date column for the carbon intensity dataset

carbon_clean["date"] = carbon_clean["from"].dt.date
carbon_clean["date"] = pd.to_datetime(carbon_clean["date"])

print("Carbon intensity columns after creating date column:")
print(carbon_clean.dtypes)

carbon_clean.head()

Carbon intensity columns after creating date column:
from                  datetime64[ns, UTC]
to                    datetime64[ns, UTC]
forecast_intensity                  int64
actual_intensity                  float64
intensity_index                    object
date                       datetime64[ns]
dtype: object


,from,to,forecast_intensity,actual_intensity,intensity_index,date
0,2023-01-01 00:00:00+00:00,2023-01-01 00:30:00+00:00,73,72.0,low,2023-01-01
1,2023-01-01 00:30:00+00:00,2023-01-01 01:00:00+00:00,63,80.0,low,2023-01-01
2,2023-01-01 01:00:00+00:00,2023-01-01 01:30:00+00:00,71,72.0,low,2023-01-01
3,2023-01-01 01:30:00+00:00,2023-01-01 02:00:00+00:00,76,65.0,low,2023-01-01
4,2023-01-01 02:00:00+00:00,2023-01-01 02:30:00+00:00,72,65.0,low,2023-01-01


In [59]:
# Create a daily carbon intensity dataset

daily_carbon = carbon_clean.groupby("date").agg({
    "forecast_intensity": "mean",
    "actual_intensity": "mean"
}).reset_index()

daily_carbon = daily_carbon.rename(columns={
    "forecast_intensity": "daily_average_forecast_carbon_intensity",
    "actual_intensity": "daily_average_actual_carbon_intensity"
})

print("Daily carbon intensity data:")
print(daily_carbon.shape)

daily_carbon.head()

Daily carbon intensity data:
(364, 3)


,date,daily_average_forecast_carbon_intensity,daily_average_actual_carbon_intensity
0,2023-01-01,106.583333,107.895833
1,2023-01-02,164.104167,157.062500
2,2023-01-03,100.500000,101.604167
3,2023-01-04,65.166667,65.500000
4,2023-01-05,103.833333,105.229167


## 5. Check Dataset Coverage and Missing Values

This section checks the date coverage, shape and missing values for the daily carbon intensity dataset, the NESO daily dataset and the ONS gas price dataset. This confirms whether each source dataset is suitable for integration.

In [60]:
# Identify missing dates in the daily carbon intensity dataset

expected_2023_dates = pd.date_range(
    start="2023-01-01",
    end="2023-12-31",
    freq="D"
)

carbon_dates = daily_carbon["date"]

missing_carbon_dates = expected_2023_dates.difference(carbon_dates)

print("Expected daily dates:", len(expected_2023_dates))
print("Actual daily carbon dates:", len(carbon_dates))
print("Missing daily carbon dates:", len(missing_carbon_dates))

missing_carbon_dates

Expected daily dates: 365
Actual daily carbon dates: 364
Missing daily carbon dates: 1


DatetimeIndex(['2023-10-21'], dtype='datetime64[ns]', freq='D')

The daily carbon intensity dataset contains 364 dates rather than 365. The missing date is 21 October 2023, which matches the earlier data source review where missing half-hourly API records were identified between 20 October and 22 October 2023. This missing date will be handled during the integration stage.

In [61]:
# Check missing values in the daily carbon intensity dataset

print("Missing values in daily carbon intensity data:")
print(daily_carbon.isna().sum())

daily_carbon[daily_carbon.isna().any(axis=1)]

Missing values in daily carbon intensity data:
date                                       0
daily_average_forecast_carbon_intensity    0
daily_average_actual_carbon_intensity      0
dtype: int64


,date,daily_average_forecast_carbon_intensity,daily_average_actual_carbon_intensity


The daily carbon intensity dataset does not contain missing values in the dates that are present. Although 21 October 2023 is missing from the Carbon Intensity API data, the remaining daily records all contain valid forecast and actual carbon intensity averages.

In [62]:
# Check NESO daily dataset coverage and missing values

print("NESO daily date range:")
print("Start date:", neso_daily["date"].min())
print("End date:", neso_daily["date"].max())

print("\nNESO daily shape:")
print(neso_daily.shape)

print("\nMissing values in NESO daily data:")
print(neso_daily.isna().sum())

neso_daily.head()

NESO daily date range:
Start date: 2023-01-01 00:00:00
End date: 2023-12-31 00:00:00

NESO daily shape:
(365, 4)

Missing values in NESO daily data:
date                0
national_demand     0
wind_generation     0
solar_generation    0
dtype: int64


,date,national_demand,wind_generation,solar_generation
0,2023-01-01,24189.979167,1733.208333,245.729167
1,2023-01-02,27005.520833,1122.416667,733.750000
2,2023-01-03,29646.312500,2927.687500,92.479167
3,2023-01-04,27967.145833,3670.312500,336.729167
4,2023-01-05,29392.500000,2528.229167,228.479167


The NESO daily dataset covers the full 2023 calendar year, from 1 January 2023 to 31 December 2023. It contains 365 daily records and no missing values in the selected demand, wind generation or solar generation variables.

In [63]:
# Check ONS gas dataset coverage and missing values

print("ONS gas date range:")
print("Start date:", gas_clean["date"].min())
print("End date:", gas_clean["date"].max())

print("\nONS gas shape:")
print(gas_clean.shape)

print("\nMissing values in ONS gas data:")
print(gas_clean.isna().sum())

gas_clean.head()

ONS gas date range:
Start date: 2023-01-01 00:00:00
End date: 2023-12-17 00:00:00

ONS gas shape:
(351, 3)

Missing values in ONS gas data:
date                                  0
sap_actual_p_per_kwh                  0
sap_7day_rolling_average_p_per_kwh    0
dtype: int64


,date,sap_actual_p_per_kwh,sap_7day_rolling_average_p_per_kwh
0,2023-01-01,5.7764,5.9602
1,2023-01-02,5.9978,5.9704
2,2023-01-03,5.6898,5.9326
3,2023-01-04,5.0746,5.8222
4,2023-01-05,5.0837,5.7133


The ONS gas price dataset covers the period from 1 January 2023 to 17 December 2023. It contains 351 daily records and no missing values in the selected gas price variables. The dataset does not cover the final two weeks of December 2023, so the integrated modelling dataset will need to account for this difference in date coverage.

## 6. Missing Data Handling Decision

This section documents how missing data issues identified during the data source review were handled before creating the final integrated dataset.

The carbon intensity dataset had two missing data issues. First, 91 half-hourly timestamps were missing from the Carbon Intensity API data, concentrated between 20 October and 22 October 2023. A separate API request was made for this period during the data source review stage, but the missing records were not returned, suggesting that the gap was due to the source API data rather than the collection code.

When the carbon intensity data was aggregated to daily level, 21 October 2023 was the only full daily date missing from the carbon dataset. The integrated dataset was created using an inner join on the date column, meaning that only dates available in all three datasets were retained. As a result, 21 October 2023 was excluded across the full integrated dataset.

Second, five half-hourly records on 26 January 2023 had missing actual carbon intensity values. These rows were not removed because the date still contained valid half-hourly observations. During daily aggregation, pandas calculated the daily average using the available actual intensity values, so the final daily carbon dataset did not contain missing daily average values.

The ONS gas dataset also ended on 17 December 2023, so dates from 18 December to 31 December 2023 were excluded from the integrated dataset through the inner join. This ensures that the final integrated dataset uses a consistent date range across all variables.

## 7. Merge the Daily Datasets

This section merges the NESO daily dataset, the daily carbon intensity dataset and the ONS gas price dataset using the date column. An inner join is used so that only dates available across all three datasets are retained.

In [64]:
# Merge the daily datasets into one integrated dataset

integrated_data = neso_daily.merge(
    daily_carbon,
    on="date",
    how="inner"
)

integrated_data = integrated_data.merge(
    gas_clean,
    on="date",
    how="inner"
)

print("Integrated dataset shape:")
print(integrated_data.shape)

print("\nIntegrated dataset date range:")
print("Start date:", integrated_data["date"].min())
print("End date:", integrated_data["date"].max())

integrated_data.head()

Integrated dataset shape:
(350, 8)

Integrated dataset date range:
Start date: 2023-01-01 00:00:00
End date: 2023-12-17 00:00:00


,date,national_demand,wind_generation,solar_generation,daily_average_forecast_carbon_intensity,daily_average_actual_carbon_intensity,sap_actual_p_per_kwh,sap_7day_rolling_average_p_per_kwh
0,2023-01-01,24189.979167,1733.208333,245.729167,106.583333,107.895833,5.7764,5.9602
1,2023-01-02,27005.520833,1122.416667,733.750000,164.104167,157.062500,5.9978,5.9704
2,2023-01-03,29646.312500,2927.687500,92.479167,100.500000,101.604167,5.6898,5.9326
3,2023-01-04,27967.145833,3670.312500,336.729167,65.166667,65.500000,5.0746,5.8222
4,2023-01-05,29392.500000,2528.229167,228.479167,103.833333,105.229167,5.0837,5.7133


The three daily datasets were merged using an inner join on the date column. This means that only dates available in all three datasets were retained. The final integrated dataset contains 350 daily records from 1 January 2023 to 17 December 2023. The date range is limited by the ONS gas price dataset, which ends on 17 December 2023, and the missing Carbon Intensity API date on 21 October 2023 is also excluded from the integrated dataset.

In [65]:
# Check missing values in the integrated dataset

print("Missing values in integrated dataset:")
print(integrated_data.isna().sum())

integrated_data[integrated_data.isna().any(axis=1)]

Missing values in integrated dataset:
date                                       0
national_demand                            0
wind_generation                            0
solar_generation                           0
daily_average_forecast_carbon_intensity    0
daily_average_actual_carbon_intensity      0
sap_actual_p_per_kwh                       0
sap_7day_rolling_average_p_per_kwh         0
dtype: int64


,date,national_demand,wind_generation,solar_generation,daily_average_forecast_carbon_intensity,daily_average_actual_carbon_intensity,sap_actual_p_per_kwh,sap_7day_rolling_average_p_per_kwh


The integrated dataset does not contain any missing values after merging. This confirms that the inner join successfully retained only the dates where NESO demand and renewable generation data, Carbon Intensity API data and ONS gas price data are all available.

## 8. Feature Engineering

This section creates additional variables that may support exploratory analysis and later modelling. These features are based on the integrated daily dataset and include time-based variables, seasonal indicators, renewable generation measures and market pressure indicators.

### 8.1 Time-Based Features

This subsection creates time-based variables from the date column, including day, month, weekday, weekday number and a weekend indicator.

In [66]:
# Create time-based features for analysis and modelling

integrated_data["day"] = integrated_data["date"].dt.day
integrated_data["month"] = integrated_data["date"].dt.month
integrated_data["weekday"] = integrated_data["date"].dt.day_name()
integrated_data["weekday_number"] = integrated_data["date"].dt.weekday
integrated_data["is_weekend"] = integrated_data["weekday_number"].isin([5, 6])

print("Integrated dataset shape after adding time-based features:")
print(integrated_data.shape)

integrated_data.head()

Integrated dataset shape after adding time-based features:
(350, 13)


,date,national_demand,wind_generation,solar_generation,daily_average_forecast_carbon_intensity,daily_average_actual_carbon_intensity,sap_actual_p_per_kwh,sap_7day_rolling_average_p_per_kwh,day,month,weekday,weekday_number,is_weekend
0,2023-01-01,24189.979167,1733.208333,245.729167,106.583333,107.895833,5.7764,5.9602,1,1,Sunday,6,True
1,2023-01-02,27005.520833,1122.416667,733.750000,164.104167,157.062500,5.9978,5.9704,2,1,Monday,0,False
2,2023-01-03,29646.312500,2927.687500,92.479167,100.500000,101.604167,5.6898,5.9326,3,1,Tuesday,1,False
3,2023-01-04,27967.145833,3670.312500,336.729167,65.166667,65.500000,5.0746,5.8222,4,1,Wednesday,2,False
4,2023-01-05,29392.500000,2528.229167,228.479167,103.833333,105.229167,5.0837,5.7133,5,1,Thursday,3,False


Additional time-based features were created from the date column. These include the day of the month, month number, weekday name, weekday number and a weekend indicator. These variables will support later exploratory analysis and modelling because electricity demand, renewable generation, carbon intensity and gas price conditions may vary across weekdays, weekends and seasons.

### 8.2 Seasonal Feature

This subsection creates a season variable from the month column. This allows later analysis to compare demand, renewable generation, carbon intensity and gas price conditions across Winter, Spring, Summer and Autumn.

In [67]:
# Create a season variable from the month column

def assign_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

integrated_data["season"] = integrated_data["month"].apply(assign_season)

print("Season counts:")
print(integrated_data["season"].value_counts())

integrated_data[["date", "month", "season"]].head()

Season counts:
season
Spring    92
Summer    92
Autumn    90
Winter    76
Name: count, dtype: int64


,date,month,season
0,2023-01-01,1,Winter
1,2023-01-02,1,Winter
2,2023-01-03,1,Winter
3,2023-01-04,1,Winter
4,2023-01-05,1,Winter


### 8.3 Renewable Generation Features

This subsection creates renewable generation features from wind generation, solar generation and national demand. These variables measure the contribution of wind and solar generation relative to electricity demand.

In [68]:
# Create renewable generation features

integrated_data["total_renewable_generation"] = (
    integrated_data["wind_generation"] + integrated_data["solar_generation"]
)

integrated_data["renewable_to_demand_ratio"] = (
    integrated_data["total_renewable_generation"] / integrated_data["national_demand"]
)

integrated_data["wind_to_demand_ratio"] = (
    integrated_data["wind_generation"] / integrated_data["national_demand"]
)

integrated_data["solar_to_demand_ratio"] = (
    integrated_data["solar_generation"] / integrated_data["national_demand"]
)

print("Integrated dataset shape after adding renewable features:")
print(integrated_data.shape)

integrated_data[
    [
        "date",
        "national_demand",
        "wind_generation",
        "solar_generation",
        "total_renewable_generation",
        "renewable_to_demand_ratio",
        "wind_to_demand_ratio",
        "solar_to_demand_ratio"
    ]
].head()

Integrated dataset shape after adding renewable features:
(350, 18)


,date,national_demand,wind_generation,solar_generation,total_renewable_generation,renewable_to_demand_ratio,wind_to_demand_ratio,solar_to_demand_ratio
0,2023-01-01,24189.979167,1733.208333,245.729167,1978.937500,0.081808,0.071650,0.010158
1,2023-01-02,27005.520833,1122.416667,733.750000,1856.166667,0.068733,0.041562,0.027170
2,2023-01-03,29646.312500,2927.687500,92.479167,3020.166667,0.101873,0.098754,0.003119
3,2023-01-04,27967.145833,3670.312500,336.729167,4007.041667,0.143277,0.131237,0.012040
4,2023-01-05,29392.500000,2528.229167,228.479167,2756.708333,0.093790,0.086016,0.007773


### 8.4 Gas Price Pressure Features

This subsection creates gas price pressure features using lagged gas price values and recent changes in the System Average Price of gas. These features may help represent recent gas market pressure in later modelling.

In [69]:
# Create gas price pressure features

integrated_data["sap_price_1_day_lag"] = integrated_data["sap_actual_p_per_kwh"].shift(1)
integrated_data["sap_price_7_day_lag"] = integrated_data["sap_actual_p_per_kwh"].shift(7)

integrated_data["sap_price_7_day_change"] = (
    integrated_data["sap_actual_p_per_kwh"] - integrated_data["sap_price_7_day_lag"]
)

print("Integrated dataset shape after adding gas price features:")
print(integrated_data.shape)

integrated_data[
    [
        "date",
        "sap_actual_p_per_kwh",
        "sap_7day_rolling_average_p_per_kwh",
        "sap_price_1_day_lag",
        "sap_price_7_day_lag",
        "sap_price_7_day_change"
    ]
].head(10)

Integrated dataset shape after adding gas price features:
(350, 21)


,date,sap_actual_p_per_kwh,sap_7day_rolling_average_p_per_kwh,sap_price_1_day_lag,sap_price_7_day_lag,sap_price_7_day_change
0,2023-01-01,5.7764,5.9602,NaN,NaN,NaN
1,2023-01-02,5.9978,5.9704,5.7764,NaN,NaN
2,2023-01-03,5.6898,5.9326,5.9978,NaN,NaN
3,2023-01-04,5.0746,5.8222,5.6898,NaN,NaN
4,2023-01-05,5.0837,5.7133,5.0746,NaN,NaN
5,2023-01-06,5.1781,5.6238,5.0837,NaN,NaN
6,2023-01-07,5.6533,5.5300,5.1781,NaN,NaN
7,2023-01-08,5.8583,5.4934,5.6533,5.7764,0.0819
8,2023-01-09,5.8128,5.5051,5.8583,5.9978,-0.1850
9,2023-01-10,5.6208,5.4787,5.8128,5.6898,-0.0690


### 8.5 Carbon Intensity Lag and Rolling Features

This subsection creates lagged and rolling carbon intensity features. These variables are useful for modelling because recent carbon intensity levels may help explain or forecast future electricity system conditions.

In [70]:
# Create carbon intensity lag and rolling features

integrated_data["actual_carbon_intensity_1_day_lag"] = (
    integrated_data["daily_average_actual_carbon_intensity"].shift(1)
)

integrated_data["actual_carbon_intensity_7_day_lag"] = (
    integrated_data["daily_average_actual_carbon_intensity"].shift(7)
)

integrated_data["actual_carbon_intensity_7_day_rolling_average"] = (
    integrated_data["daily_average_actual_carbon_intensity"]
    .shift(1)
    .rolling(window=7)
    .mean()
)

integrated_data["forecast_carbon_intensity_1_day_lag"] = (
    integrated_data["daily_average_forecast_carbon_intensity"].shift(1)
)

print("Integrated dataset shape after adding carbon intensity features:")
print(integrated_data.shape)

integrated_data[
    [
        "date",
        "daily_average_actual_carbon_intensity",
        "actual_carbon_intensity_1_day_lag",
        "actual_carbon_intensity_7_day_lag",
        "actual_carbon_intensity_7_day_rolling_average",
        "daily_average_forecast_carbon_intensity",
        "forecast_carbon_intensity_1_day_lag"
    ]
].head(10)

Integrated dataset shape after adding carbon intensity features:
(350, 25)


,date,daily_average_actual_carbon_intensity,actual_carbon_intensity_1_day_lag,actual_carbon_intensity_7_day_lag,actual_carbon_intensity_7_day_rolling_average,daily_average_forecast_carbon_intensity,forecast_carbon_intensity_1_day_lag
0,2023-01-01,107.895833,NaN,NaN,NaN,106.583333,NaN
1,2023-01-02,157.062500,107.895833,NaN,NaN,164.104167,106.583333
2,2023-01-03,101.604167,157.062500,NaN,NaN,100.500000,164.104167
3,2023-01-04,65.500000,101.604167,NaN,NaN,65.166667,100.500000
4,2023-01-05,105.229167,65.500000,NaN,NaN,103.833333,65.166667
5,2023-01-06,85.041667,105.229167,NaN,NaN,84.166667,103.833333
6,2023-01-07,64.479167,85.041667,NaN,NaN,61.333333,84.166667
7,2023-01-08,69.666667,64.479167,107.895833,98.116071,69.083333,61.333333
8,2023-01-09,83.833333,69.666667,157.062500,92.654762,77.958333,69.083333
9,2023-01-10,81.125000,83.833333,101.604167,82.193452,78.500000,77.958333


Carbon intensity lag and rolling features were created because the project is a forecasting problem. These features allow later models to use recent carbon intensity behaviour, such as the previous day’s value, the value from seven days earlier and the previous seven-day average, without using the current day’s actual value directly. This helps reduce data leakage and gives the model information about short-term and weekly patterns in carbon intensity.

### 8.6 Demand and Renewable Lag Features

This subsection creates lagged demand and renewable generation features. These variables are useful because electricity demand and renewable generation conditions often show short-term persistence over time.

In [71]:
# Create demand and renewable lag features

integrated_data["national_demand_1_day_lag"] = (
    integrated_data["national_demand"].shift(1)
)

integrated_data["national_demand_7_day_lag"] = (
    integrated_data["national_demand"].shift(7)
)

integrated_data["total_renewable_generation_1_day_lag"] = (
    integrated_data["total_renewable_generation"].shift(1)
)

integrated_data["total_renewable_generation_7_day_lag"] = (
    integrated_data["total_renewable_generation"].shift(7)
)

integrated_data["renewable_to_demand_ratio_1_day_lag"] = (
    integrated_data["renewable_to_demand_ratio"].shift(1)
)

integrated_data["renewable_to_demand_ratio_7_day_lag"] = (
    integrated_data["renewable_to_demand_ratio"].shift(7)
)

print("Integrated dataset shape after adding demand and renewable lag features:")
print(integrated_data.shape)

integrated_data[
    [
        "date",
        "national_demand",
        "national_demand_1_day_lag",
        "national_demand_7_day_lag",
        "total_renewable_generation",
        "total_renewable_generation_1_day_lag",
        "total_renewable_generation_7_day_lag",
        "renewable_to_demand_ratio",
        "renewable_to_demand_ratio_1_day_lag",
        "renewable_to_demand_ratio_7_day_lag"
    ]
].head(10)

Integrated dataset shape after adding demand and renewable lag features:
(350, 31)


,date,national_demand,national_demand_1_day_lag,national_demand_7_day_lag,total_renewable_generation,total_renewable_generation_1_day_lag,total_renewable_generation_7_day_lag,renewable_to_demand_ratio,renewable_to_demand_ratio_1_day_lag,renewable_to_demand_ratio_7_day_lag
0,2023-01-01,24189.979167,NaN,NaN,1978.937500,NaN,NaN,0.081808,NaN,NaN
1,2023-01-02,27005.520833,24189.979167,NaN,1856.166667,1978.937500,NaN,0.068733,0.081808,NaN
2,2023-01-03,29646.312500,27005.520833,NaN,3020.166667,1856.166667,NaN,0.101873,0.068733,NaN
3,2023-01-04,27967.145833,29646.312500,NaN,4007.041667,3020.166667,NaN,0.143277,0.101873,NaN
4,2023-01-05,29392.500000,27967.145833,NaN,2756.708333,4007.041667,NaN,0.093790,0.143277,NaN
5,2023-01-06,28316.125000,29392.500000,NaN,3566.250000,2756.708333,NaN,0.125944,0.093790,NaN
6,2023-01-07,25280.937500,28316.125000,NaN,3681.041667,3566.250000,NaN,0.145605,0.125944,NaN
7,2023-01-08,26629.479167,25280.937500,24189.979167,3391.541667,3681.041667,1978.937500,0.127360,0.145605,0.081808
8,2023-01-09,30403.104167,26629.479167,27005.520833,3418.500000,3391.541667,1856.166667,0.112439,0.127360,0.068733
9,2023-01-10,31143.895833,30403.104167,29646.312500,3530.041667,3418.500000,3020.166667,0.113346,0.112439,0.101873


Demand and renewable lag features were created to capture short-term and weekly persistence in electricity demand and renewable generation. The one-day lag features represent the previous available daily observation, while the seven-day lag features represent conditions from seven previous observations. These variables may help later forecasting models because demand and renewable generation patterns often depend on recent system conditions.

## 9. Final Modelling Dataset Check

This section checks the final feature-engineered dataset before it is saved. The checks include the final shape, column list and missing values created by lag and rolling features.

In [72]:
# Check the final feature-engineered dataset and create a modelling-ready version

print("Feature-engineered integrated dataset shape:")
print(integrated_data.shape)

print("\nColumns in the feature-engineered dataset:")
print(integrated_data.columns.tolist())

print("\nMissing values in the feature-engineered dataset:")
missing_values = integrated_data.isna().sum()
print(missing_values[missing_values > 0])

model_ready_data = integrated_data.dropna().copy()
model_ready_data = model_ready_data.reset_index(drop=True)

print("\nModel-ready dataset shape after removing rows with lag-related missing values:")
print(model_ready_data.shape)

print("\nModel-ready dataset date range:")
print("Start date:", model_ready_data["date"].min())
print("End date:", model_ready_data["date"].max())

print("\nRemaining missing values in model-ready dataset:")
print(model_ready_data.isna().sum().sum())

model_ready_data.head()

Feature-engineered integrated dataset shape:
(350, 31)

Columns in the feature-engineered dataset:
['date', 'national_demand', 'wind_generation', 'solar_generation', 'daily_average_forecast_carbon_intensity', 'daily_average_actual_carbon_intensity', 'sap_actual_p_per_kwh', 'sap_7day_rolling_average_p_per_kwh', 'day', 'month', 'weekday', 'weekday_number', 'is_weekend', 'season', 'total_renewable_generation', 'renewable_to_demand_ratio', 'wind_to_demand_ratio', 'solar_to_demand_ratio', 'sap_price_1_day_lag', 'sap_price_7_day_lag', 'sap_price_7_day_change', 'actual_carbon_intensity_1_day_lag', 'actual_carbon_intensity_7_day_lag', 'actual_carbon_intensity_7_day_rolling_average', 'forecast_carbon_intensity_1_day_lag', 'national_demand_1_day_lag', 'national_demand_7_day_lag', 'total_renewable_generation_1_day_lag', 'total_renewable_generation_7_day_lag', 'renewable_to_demand_ratio_1_day_lag', 'renewable_to_demand_ratio_7_day_lag']

Missing values in the feature-engineered dataset:
sap_price_

,date,national_demand,wind_generation,solar_generation,daily_average_forecast_carbon_intensity,daily_average_actual_carbon_intensity,sap_actual_p_per_kwh,sap_7day_rolling_average_p_per_kwh,day,month,...,actual_carbon_intensity_1_day_lag,actual_carbon_intensity_7_day_lag,actual_carbon_intensity_7_day_rolling_average,forecast_carbon_intensity_1_day_lag,national_demand_1_day_lag,national_demand_7_day_lag,total_renewable_generation_1_day_lag,total_renewable_generation_7_day_lag,renewable_to_demand_ratio_1_day_lag,renewable_to_demand_ratio_7_day_lag
0,2023-01-08,26629.479167,3074.645833,316.895833,69.083333,69.666667,5.8583,5.4934,8,1,...,64.479167,107.895833,98.116071,61.333333,25280.937500,24189.979167,3681.041667,1978.937500,0.145605,0.081808
1,2023-01-09,30403.104167,2832.229167,586.270833,77.958333,83.833333,5.8128,5.5051,9,1,...,69.666667,157.062500,92.654762,69.083333,26629.479167,27005.520833,3391.541667,1856.166667,0.127360,0.068733
2,2023-01-10,31143.895833,3470.000000,60.041667,78.500000,81.125000,5.6208,5.4787,10,1,...,83.833333,101.604167,82.193452,77.958333,30403.104167,29646.312500,3418.500000,3020.166667,0.112439,0.101873
3,2023-01-11,29849.687500,3869.895833,452.208333,67.729167,71.479167,5.4858,5.4688,11,1,...,81.125000,65.500000,79.267857,78.500000,31143.895833,27967.145833,3530.041667,4007.041667,0.113346,0.143277
4,2023-01-12,30083.520833,3594.104167,184.958333,75.770833,76.812500,5.6646,5.5275,12,1,...,71.479167,105.229167,80.122024,67.729167,29849.687500,29392.500000,4322.104167,2756.708333,0.144796,0.093790


The final feature-engineered dataset contains lag and rolling features that naturally create missing values at the start of the time series. This is expected because the first few dates do not have enough previous observations to calculate one-day lags, seven-day lags or seven-day rolling averages.

A modelling-ready dataset was therefore created by removing rows with lag-related missing values. After this step, the modelling-ready dataset starts on 8 January 2023 and contains no remaining missing values. This version will be used for the initial modelling stage in the next notebook.

## 10. Save Final Outputs

This section saves the final integrated and feature-engineered dataset to the processed data folder. It also saves a summary table documenting the final dataset shape, date range and missing value count.

In [73]:
# Save the final feature-engineered and model-ready datasets

feature_engineered_data_file = (
    processed_data_folder / "integrated_energy_market_risk_2023.csv"
)

model_ready_data_file = (
    processed_data_folder / "model_ready_energy_market_risk_2023.csv"
)

integrated_data.to_csv(feature_engineered_data_file, index=False)
model_ready_data.to_csv(model_ready_data_file, index=False)

print("Saved feature-engineered integrated dataset to:")
print(feature_engineered_data_file)

print("\nSaved model-ready dataset to:")
print(model_ready_data_file)

Saved feature-engineered integrated dataset to:
/Users/tosinjimoh/Downloads/Msc/Lecture Notes/2026-MSc-Dissertation-Project/Data/processed/integrated_energy_market_risk_2023.csv

Saved model-ready dataset to:
/Users/tosinjimoh/Downloads/Msc/Lecture Notes/2026-MSc-Dissertation-Project/Data/processed/model_ready_energy_market_risk_2023.csv


In [74]:
# Read back the saved datasets to confirm they saved correctly

saved_feature_engineered_data = pd.read_csv(feature_engineered_data_file)
saved_model_ready_data = pd.read_csv(model_ready_data_file)

print("Saved feature-engineered dataset shape:")
print(saved_feature_engineered_data.shape)

print("\nSaved model-ready dataset shape:")
print(saved_model_ready_data.shape)

print("\nMissing values in saved feature-engineered dataset:")
print(saved_feature_engineered_data.isna().sum().sum())

print("\nMissing values in saved model-ready dataset:")
print(saved_model_ready_data.isna().sum().sum())

saved_model_ready_data.head()

Saved feature-engineered dataset shape:
(350, 31)

Saved model-ready dataset shape:
(343, 31)

Missing values in saved feature-engineered dataset:
55

Missing values in saved model-ready dataset:
0


,date,national_demand,wind_generation,solar_generation,daily_average_forecast_carbon_intensity,daily_average_actual_carbon_intensity,sap_actual_p_per_kwh,sap_7day_rolling_average_p_per_kwh,day,month,...,actual_carbon_intensity_1_day_lag,actual_carbon_intensity_7_day_lag,actual_carbon_intensity_7_day_rolling_average,forecast_carbon_intensity_1_day_lag,national_demand_1_day_lag,national_demand_7_day_lag,total_renewable_generation_1_day_lag,total_renewable_generation_7_day_lag,renewable_to_demand_ratio_1_day_lag,renewable_to_demand_ratio_7_day_lag
0,2023-01-08,26629.479167,3074.645833,316.895833,69.083333,69.666667,5.8583,5.4934,8,1,...,64.479167,107.895833,98.116071,61.333333,25280.937500,24189.979167,3681.041667,1978.937500,0.145605,0.081808
1,2023-01-09,30403.104167,2832.229167,586.270833,77.958333,83.833333,5.8128,5.5051,9,1,...,69.666667,157.062500,92.654762,69.083333,26629.479167,27005.520833,3391.541667,1856.166667,0.127360,0.068733
2,2023-01-10,31143.895833,3470.000000,60.041667,78.500000,81.125000,5.6208,5.4787,10,1,...,83.833333,101.604167,82.193452,77.958333,30403.104167,29646.312500,3418.500000,3020.166667,0.112439,0.101873
3,2023-01-11,29849.687500,3869.895833,452.208333,67.729167,71.479167,5.4858,5.4688,11,1,...,81.125000,65.500000,79.267857,78.500000,31143.895833,27967.145833,3530.041667,4007.041667,0.113346,0.143277
4,2023-01-12,30083.520833,3594.104167,184.958333,75.770833,76.812500,5.6646,5.5275,12,1,...,71.479167,105.229167,80.122024,67.729167,29849.687500,29392.500000,4322.104167,2756.708333,0.144796,0.093790


In [75]:
# Create and save a summary table for the final datasets

final_dataset_summary = pd.DataFrame([
    {
        "dataset": "Feature-engineered integrated dataset",
        "file_name": "integrated_energy_market_risk_2023.csv",
        "rows": integrated_data.shape[0],
        "columns": integrated_data.shape[1],
        "start_date": integrated_data["date"].min(),
        "end_date": integrated_data["date"].max(),
        "missing_values": integrated_data.isna().sum().sum(),
        "notes": "Full feature-engineered dataset. Lag-related missing values are retained at the start of the time series."
    },
    {
        "dataset": "Model-ready dataset",
        "file_name": "model_ready_energy_market_risk_2023.csv",
        "rows": model_ready_data.shape[0],
        "columns": model_ready_data.shape[1],
        "start_date": model_ready_data["date"].min(),
        "end_date": model_ready_data["date"].max(),
        "missing_values": model_ready_data.isna().sum().sum(),
        "notes": "Rows with lag-related missing values removed. This dataset will be used for initial modelling."
    }
])

final_dataset_summary_file = tables_folder / "final_dataset_summary.csv"
final_dataset_summary.to_csv(final_dataset_summary_file, index=False)

print("Saved final dataset summary table to:")
print(final_dataset_summary_file)

final_dataset_summary

Saved final dataset summary table to:
/Users/tosinjimoh/Downloads/Msc/Lecture Notes/2026-MSc-Dissertation-Project/Outputs/tables/final_dataset_summary.csv


,dataset,file_name,rows,columns,start_date,end_date,missing_values,notes
0,Feature-engineered integrated dataset,integrated_energy_market_risk_2023.csv,350,31,2023-01-01,2023-12-17,55,Full feature-engineered dataset. Lag-related m...
1,Model-ready dataset,model_ready_energy_market_risk_2023.csv,343,31,2023-01-08,2023-12-17,0,Rows with lag-related missing values removed. ...


The final dataset summary table records both saved datasets. The full feature-engineered dataset contains 350 rows and 31 columns, but retains lag-related missing values at the start of the time series. The model-ready dataset contains 343 rows and 31 columns after removing rows with lag-related missing values. This model-ready dataset contains no missing values and will be used in the next notebook for exploratory analysis and initial modelling.

In [76]:
# Create and save a data dictionary for the final model-ready dataset

final_data_dictionary = pd.DataFrame([
    {
        "column_name": "date",
        "description": "Daily observation date used to align all datasets.",
        "unit_or_measurement": "Date"
    },
    {
        "column_name": "national_demand",
        "description": "Daily average national electricity demand.",
        "unit_or_measurement": "MW"
    },
    {
        "column_name": "wind_generation",
        "description": "Daily average embedded wind generation.",
        "unit_or_measurement": "MW"
    },
    {
        "column_name": "solar_generation",
        "description": "Daily average embedded solar generation.",
        "unit_or_measurement": "MW"
    },
    {
        "column_name": "daily_average_forecast_carbon_intensity",
        "description": "Daily average forecast carbon intensity.",
        "unit_or_measurement": "gCO2/kWh"
    },
    {
        "column_name": "daily_average_actual_carbon_intensity",
        "description": "Daily average actual carbon intensity.",
        "unit_or_measurement": "gCO2/kWh"
    },
    {
        "column_name": "sap_actual_p_per_kwh",
        "description": "Actual daily System Average Price of gas.",
        "unit_or_measurement": "p/kWh"
    },
    {
        "column_name": "sap_7day_rolling_average_p_per_kwh",
        "description": "Preceding seven-day rolling average of the System Average Price of gas.",
        "unit_or_measurement": "p/kWh"
    },
    {
        "column_name": "day",
        "description": "Day of the month created from the date column.",
        "unit_or_measurement": "Day number"
    },
    {
        "column_name": "month",
        "description": "Month of the year created from the date column.",
        "unit_or_measurement": "Month number"
    },
    {
        "column_name": "weekday",
        "description": "Day name created from the date column.",
        "unit_or_measurement": "Day name"
    },
    {
        "column_name": "weekday_number",
        "description": "Numeric weekday indicator, where Monday is 0 and Sunday is 6.",
        "unit_or_measurement": "Number"
    },
    {
        "column_name": "is_weekend",
        "description": "Boolean indicator showing whether the date falls on a weekend.",
        "unit_or_measurement": "True/False"
    },
    {
        "column_name": "season",
        "description": "Season assigned from the month column.",
        "unit_or_measurement": "Category"
    },
    {
        "column_name": "total_renewable_generation",
        "description": "Combined daily average wind and solar generation.",
        "unit_or_measurement": "MW"
    },
    {
        "column_name": "renewable_to_demand_ratio",
        "description": "Total wind and solar generation divided by national demand.",
        "unit_or_measurement": "Ratio"
    },
    {
        "column_name": "wind_to_demand_ratio",
        "description": "Wind generation divided by national demand.",
        "unit_or_measurement": "Ratio"
    },
    {
        "column_name": "solar_to_demand_ratio",
        "description": "Solar generation divided by national demand.",
        "unit_or_measurement": "Ratio"
    },
    {
        "column_name": "sap_price_1_day_lag",
        "description": "Previous available daily System Average Price of gas.",
        "unit_or_measurement": "p/kWh"
    },
    {
        "column_name": "sap_price_7_day_lag",
        "description": "System Average Price of gas from seven previous observations.",
        "unit_or_measurement": "p/kWh"
    },
    {
        "column_name": "sap_price_7_day_change",
        "description": "Difference between current gas price and the gas price from seven previous observations.",
        "unit_or_measurement": "p/kWh"
    },
    {
        "column_name": "actual_carbon_intensity_1_day_lag",
        "description": "Previous available daily average actual carbon intensity.",
        "unit_or_measurement": "gCO2/kWh"
    },
    {
        "column_name": "actual_carbon_intensity_7_day_lag",
        "description": "Daily average actual carbon intensity from seven previous observations.",
        "unit_or_measurement": "gCO2/kWh"
    },
    {
        "column_name": "actual_carbon_intensity_7_day_rolling_average",
        "description": "Average actual carbon intensity over the previous seven observations.",
        "unit_or_measurement": "gCO2/kWh"
    },
    {
        "column_name": "forecast_carbon_intensity_1_day_lag",
        "description": "Previous available daily average forecast carbon intensity.",
        "unit_or_measurement": "gCO2/kWh"
    },
    {
        "column_name": "national_demand_1_day_lag",
        "description": "Previous available daily average national electricity demand.",
        "unit_or_measurement": "MW"
    },
    {
        "column_name": "national_demand_7_day_lag",
        "description": "Daily average national electricity demand from seven previous observations.",
        "unit_or_measurement": "MW"
    },
    {
        "column_name": "total_renewable_generation_1_day_lag",
        "description": "Previous available total wind and solar generation.",
        "unit_or_measurement": "MW"
    },
    {
        "column_name": "total_renewable_generation_7_day_lag",
        "description": "Total wind and solar generation from seven previous observations.",
        "unit_or_measurement": "MW"
    },
    {
        "column_name": "renewable_to_demand_ratio_1_day_lag",
        "description": "Previous available renewable-to-demand ratio.",
        "unit_or_measurement": "Ratio"
    },
    {
        "column_name": "renewable_to_demand_ratio_7_day_lag",
        "description": "Renewable-to-demand ratio from seven previous observations.",
        "unit_or_measurement": "Ratio"
    }
])

final_data_dictionary_file = tables_folder / "final_model_ready_data_dictionary.csv"
final_data_dictionary.to_csv(final_data_dictionary_file, index=False)

print("Saved final model-ready data dictionary to:")
print(final_data_dictionary_file)

print("\nData dictionary shape:")
print(final_data_dictionary.shape)

final_data_dictionary

Saved final model-ready data dictionary to:
/Users/tosinjimoh/Downloads/Msc/Lecture Notes/2026-MSc-Dissertation-Project/Outputs/tables/final_model_ready_data_dictionary.csv

Data dictionary shape:
(31, 3)


,column_name,description,unit_or_measurement
0,date,Daily observation date used to align all datas...,Date
1,national_demand,Daily average national electricity demand.,MW
2,wind_generation,Daily average embedded wind generation.,MW
3,solar_generation,Daily average embedded solar generation.,MW
4,daily_average_forecast_carbon_intensity,Daily average forecast carbon intensity.,gCO2/kWh
5,daily_average_actual_carbon_intensity,Daily average actual carbon intensity.,gCO2/kWh
6,sap_actual_p_per_kwh,Actual daily System Average Price of gas.,p/kWh
7,sap_7day_rolling_average_p_per_kwh,Preceding seven-day rolling average of the Sys...,p/kWh
8,day,Day of the month created from the date column.,Day number
9,month,Month of the year created from the date column.,Month number


A final data dictionary was created for the model-ready dataset. This records each column name, its meaning and its unit or measurement. The data dictionary supports transparency and makes it easier to explain the final modelling dataset in the dissertation and during supervisor discussions.

## 11. Notebook Summary

This notebook prepared the selected dissertation datasets for integration and initial modelling. The NESO daily demand, wind generation and solar generation dataset was loaded alongside the cleaned Carbon Intensity API dataset and the cleaned ONS System Average Price of Gas dataset.

The Carbon Intensity API data was aggregated from half-hourly observations to daily average forecast and actual carbon intensity values so that it could be aligned with the daily NESO and ONS gas datasets. Dataset coverage and missing values were checked before integration.

The main missing data issue was identified in the Carbon Intensity API data. The carbon dataset was missing 91 half-hourly timestamps, concentrated between 20 October and 22 October 2023. After daily aggregation, 21 October 2023 was the only full missing carbon intensity date. The datasets were therefore merged using an inner join on the date column, meaning only dates available across all three datasets were retained. The ONS gas dataset ended on 17 December 2023, so the final integrated date range also ends on this date.

Feature engineering was then carried out to create variables for later exploratory analysis and modelling. These included time-based variables, a seasonal feature, renewable generation ratios, gas price lag features, carbon intensity lag and rolling features, and demand and renewable lag features.

Two final datasets were saved. The full feature-engineered integrated dataset contains 350 rows and 31 columns. This dataset retains lag-related missing values at the start of the time series for transparency. A separate model-ready dataset was also created by removing rows with lag-related missing values. The model-ready dataset contains 343 rows and 31 columns, covers the period from 8 January 2023 to 17 December 2023, and contains no missing values.

The final outputs saved from this notebook are:
- `integrated_energy_market_risk_2023.csv`
- `model_ready_energy_market_risk_2023.csv`
- `final_dataset_summary.csv`
- `final_model_ready_data_dictionary.csv`

The model-ready dataset will be used in the next notebook for exploratory data analysis and initial machine learning modelling.